# Check data

In [1]:
from data import build_dataloaders

train, val, inv = build_dataloaders("../data/makeathon-challenge", cache_dir="../cache", preprocess_kwargs={"apply_lee_filter": True})


## Test vegetation predictor

In [2]:
from model import HeuristicForestMasker, sweep_ndvi_threshold, evaluate_masker_on_tiles
from pathlib import Path

paths = sorted(Path("../cache").glob("*.npz"))
rows = sweep_ndvi_threshold(paths, ndvi_grid=[0.45, 0.5, 0.55, 0.6, 0.65, 0.7])
# Find the tightest threshold that keeps recall ≥ 0.95


In [3]:
for r in rows:
    print(f"ndvi={r['ndvi_threshold']:.2f}  recall={r['recall_vs_alerts']:.3f}  "
          f"coverage={r['mask_coverage']:.3f}  alerts={r['n_alert_pixels']}")


ndvi=0.45  recall=0.184  coverage=0.089  alerts=2062570
ndvi=0.50  recall=0.184  coverage=0.088  alerts=2062570
ndvi=0.55  recall=0.184  coverage=0.086  alerts=2062570
ndvi=0.60  recall=0.183  coverage=0.082  alerts=2062570
ndvi=0.65  recall=0.181  coverage=0.077  alerts=2062570
ndvi=0.70  recall=0.177  coverage=0.072  alerts=2062570


Sanity Check

In [4]:
# 1. Lock in the chosen threshold and look at the full report
best = HeuristicForestMasker(ndvi_threshold=0.60)  # ← whatever you picked
report = evaluate_masker_on_tiles(best, paths)
print(report["aggregate"])          # recall, coverage, n_tiles
print(report["per_tile"])            # watch for outlier tiles

# 2. Compare against the learned masker
from model import LearnedForestMasker
import numpy as np
tensors_list = [dict(np.load(p)) for p in paths]
learned = LearnedForestMasker(backend="lightgbm").fit(tensors_list)
print(evaluate_masker_on_tiles(learned, paths)["aggregate"])


{'recall_vs_alerts': 0.18283403714783014, 'mask_coverage': 0.08175199992745816, 'n_alert_pixels': 2062570, 'n_tiles': 10}
{'18NWG_6_6': {'recall_vs_alerts': 0.0, 'n_alert_pixels': 369966, 'n_alert_in_mask': 0, 'mask_coverage': 0.0, 'mask_pixels': 0, 'tile_pixels': 1004004}, '18NWH_1_4': {'recall_vs_alerts': 0.022160350019887493, 'n_alert_pixels': 35198, 'n_alert_in_mask': 780, 'mask_coverage': 0.020688164588985702, 'mask_pixels': 20771, 'tile_pixels': 1004004}, '18NXH_6_8': {'recall_vs_alerts': 0.0, 'n_alert_pixels': 368038, 'n_alert_in_mask': 0, 'mask_coverage': 0.0, 'mask_pixels': 0, 'tile_pixels': 1004004}, '18NXJ_7_6': {'recall_vs_alerts': 0.0, 'n_alert_pixels': 29995, 'n_alert_in_mask': 0, 'mask_coverage': 0.0, 'mask_pixels': 0, 'tile_pixels': 1008016}, '48PUT_0_8': {'recall_vs_alerts': 0.0, 'n_alert_pixels': 74413, 'n_alert_in_mask': 0, 'mask_coverage': 0.0, 'mask_pixels': 0, 'tile_pixels': 1018072}, '48PWV_7_8': {'recall_vs_alerts': 0.66695944200465, 'n_alert_pixels': 464520, 'n

/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/sklea

{'recall_vs_alerts': 0.9997507963366091, 'mask_coverage': 0.9458966941024661, 'n_alert_pixels': 2062570, 'n_tiles': 10}


## Diagnose mask pixel issue

In [5]:
import numpy as np
for p in paths:
    t = dict(np.load(p))
    ndvi = t["s2_pre_ndvi_median"]
    print(f"{p.stem}: ndvi_pre min={ndvi.min():.3f} "
          f"max={ndvi.max():.3f} mean={ndvi.mean():.3f} "
          f"nonzero={(ndvi != 0).mean():.3f}")


18NWG_6_6: ndvi_pre min=-0.125 max=0.893 mean=0.696 nonzero=1.000
18NWH_1_4: ndvi_pre min=-0.139 max=0.888 mean=0.688 nonzero=1.000
18NWJ_8_9: ndvi_pre min=-0.058 max=0.875 mean=0.639 nonzero=1.000
18NWM_9_4: ndvi_pre min=0.000 max=0.705 mean=0.289 nonzero=0.690
18NXH_6_8: ndvi_pre min=0.000 max=0.880 mean=0.793 nonzero=1.000
18NXJ_7_6: ndvi_pre min=-0.246 max=0.881 mean=0.676 nonzero=0.997
18NYH_9_9: ndvi_pre min=0.000 max=0.208 mean=0.125 nonzero=0.750
19NBD_4_4: ndvi_pre min=0.000 max=0.892 mean=0.235 nonzero=0.994
47QMB_0_8: ndvi_pre min=0.000 max=0.869 mean=0.584 nonzero=0.991
47QQV_2_4: ndvi_pre min=-0.047 max=0.851 mean=0.511 nonzero=0.978
48PUT_0_8: ndvi_pre min=-0.220 max=0.885 mean=0.545 nonzero=0.970
48PWV_7_8: ndvi_pre min=0.000 max=0.883 mean=0.708 nonzero=0.996
48PXC_7_7: ndvi_pre min=-0.320 max=0.889 mean=0.622 nonzero=0.986
48PYB_3_6: ndvi_pre min=-0.330 max=0.899 mean=0.672 nonzero=0.982
48QVE_3_0: ndvi_pre min=-0.388 max=0.864 mean=0.583 nonzero=0.993
48QWD_2_2: ndvi_

In [6]:
from model import HeuristicForestMasker
for p in paths:
    t = dict(np.load(p))
    ndvi, nbr, ndvi_std = t["s2_pre_ndvi_median"], t["s2_pre_nbr_median"], t["s2_pre_ndvi_std"]
    g_ndvi = ndvi >= 0.6
    g_nbr  = nbr  >= 0.3
    g_std  = ndvi_std <= 0.18
    print(f"{p.stem}: ndvi_gate={g_ndvi.mean():.3f}  "
          f"nbr_gate={g_nbr.mean():.3f}  std_gate={g_std.mean():.3f}  "
          f"AND={(g_ndvi & g_nbr & g_std).mean():.3f}")


18NWG_6_6: ndvi_gate=0.797  nbr_gate=0.897  std_gate=0.022  AND=0.001
18NWH_1_4: ndvi_gate=0.787  nbr_gate=0.961  std_gate=0.107  AND=0.036
18NWJ_8_9: ndvi_gate=0.660  nbr_gate=0.990  std_gate=0.003  AND=0.000
18NWM_9_4: ndvi_gate=0.087  nbr_gate=0.627  std_gate=0.344  AND=0.000
18NXH_6_8: ndvi_gate=0.982  nbr_gate=0.989  std_gate=0.001  AND=0.000
18NXJ_7_6: ndvi_gate=0.684  nbr_gate=0.848  std_gate=0.099  AND=0.002
18NYH_9_9: ndvi_gate=0.000  nbr_gate=0.750  std_gate=0.250  AND=0.000
19NBD_4_4: ndvi_gate=0.274  nbr_gate=0.279  std_gate=0.005  AND=0.000
47QMB_0_8: ndvi_gate=0.523  nbr_gate=0.963  std_gate=0.013  AND=0.000
47QQV_2_4: ndvi_gate=0.333  nbr_gate=0.719  std_gate=0.072  AND=0.000
48PUT_0_8: ndvi_gate=0.485  nbr_gate=0.807  std_gate=0.068  AND=0.000
48PWV_7_8: ndvi_gate=0.800  nbr_gate=0.814  std_gate=0.676  AND=0.562
48PXC_7_7: ndvi_gate=0.671  nbr_gate=0.684  std_gate=0.436  AND=0.244
48PYB_3_6: ndvi_gate=0.771  nbr_gate=0.798  std_gate=0.074  AND=0.017
48QVE_3_0: ndvi_gate

## std_gate is underperforming... -> Excluding it

In [7]:
from model import HeuristicForestMasker, evaluate_masker_on_tiles

masker = HeuristicForestMasker(
    ndvi_threshold=0.6,
    nbr_threshold=0.3,
    max_ndvi_std=None,          # was 0.18 — too tight
    min_component_pixels=100,
)
print(evaluate_masker_on_tiles(masker, paths)["aggregate"])


{'recall_vs_alerts': 0.9559496162554483, 'mask_coverage': 0.6912936379617791, 'n_alert_pixels': 2062570, 'n_tiles': 10}


In [8]:
rows = sweep_ndvi_threshold(paths,
    ndvi_grid=[0.45, 0.50, 0.55, 0.60, 0.65, 0.70],
    base_masker_kwargs={"max_ndvi_std": None, "nbr_threshold": 0.3})


In [10]:
for r in rows:
    print(f"ndvi={r['ndvi_threshold']:.2f}  recall={r['recall_vs_alerts']:.3f}  "
          f"coverage={r['mask_coverage']:.3f}  alerts={r['n_alert_pixels']}")


ndvi=0.45  recall=0.985  coverage=0.788  alerts=2062570
ndvi=0.50  recall=0.981  coverage=0.772  alerts=2062570
ndvi=0.55  recall=0.973  coverage=0.743  alerts=2062570
ndvi=0.60  recall=0.956  coverage=0.691  alerts=2062570
ndvi=0.65  recall=0.922  coverage=0.622  alerts=2062570
ndvi=0.70  recall=0.849  coverage=0.534  alerts=2062570


In [11]:
best = HeuristicForestMasker(ndvi_threshold=0.60, nbr_threshold=0.3, max_ndvi_std=None)
report = evaluate_masker_on_tiles(best, paths)
for tid, m in report["per_tile"].items():
    print(f"{tid}: recall={m['recall_vs_alerts']:.3f}  coverage={m['mask_coverage']:.3f}")


18NWG_6_6: recall=0.932  coverage=0.789
18NWH_1_4: recall=0.857  coverage=0.778
18NXH_6_8: recall=0.997  coverage=0.981
18NXJ_7_6: recall=0.997  coverage=0.662
48PUT_0_8: recall=0.754  coverage=0.468
48PWV_7_8: recall=0.989  coverage=0.789
48PXC_7_7: recall=0.962  coverage=0.630
48PYB_3_6: recall=0.959  coverage=0.753
48QVE_3_0: recall=0.933  coverage=0.548
48QWD_2_2: recall=0.944  coverage=0.519


### Aggregation looks good, but 48PUT_0_8 drops to ecall=0.754 -> ~25% of alerts will be dropped down-stream

In [12]:
t = dict(np.load("../cache/48PUT_0_8.npz"))
missed = (t["forest_gt_pre2020"] > 0) & (t["s2_pre_ndvi_median"] < 0.6)
print(f"missed alert pixels: {missed.sum():,}  "
      f"their ndvi median: {t['s2_pre_ndvi_median'][missed].mean():.3f}")


missed alert pixels: 17,573  their ndvi median: 0.423


### On 48PUT_0_8 17,573 missed pixels had NDVI 0.42 -> something was there post 2020, but at NDVI 0.42 they look identical to pasture -> No single NDVI threshold can separate them.

In [13]:
import numpy as np
from pathlib import Path
from model import HeuristicForestMasker, LearnedForestMasker, evaluate_masker_on_tiles

paths = sorted(Path("../cache").glob("*.npz"))
tensors_list = [dict(np.load(p)) for p in paths]

heur = HeuristicForestMasker(ndvi_threshold=0.55, nbr_threshold=0.3, max_ndvi_std=None)
lgbm = LearnedForestMasker(backend="lightgbm").fit(tensors_list)

h = evaluate_masker_on_tiles(heur, paths)
l = evaluate_masker_on_tiles(lgbm, paths)

print(f"{'tile':<14} {'heur_r':>7} {'heur_c':>7}  {'lgbm_r':>7} {'lgbm_c':>7}")
for tid in h["per_tile"]:
    hp, lp = h["per_tile"][tid], l["per_tile"][tid]
    print(f"{tid:<14} {hp['recall_vs_alerts']:>7.3f} {hp['mask_coverage']:>7.3f}  "
          f"{lp['recall_vs_alerts']:>7.3f} {lp['mask_coverage']:>7.3f}")

print(f"\nHeuristic agg: recall={h['aggregate']['recall_vs_alerts']:.3f}  "
      f"coverage={h['aggregate']['mask_coverage']:.3f}")
print(f"Learned   agg: recall={l['aggregate']['recall_vs_alerts']:.3f}  "
      f"coverage={l['aggregate']['mask_coverage']:.3f}")

# Sanity on the weak tile: does lgbm rescue the pixels heuristic missed?
t = dict(np.load("../cache/48PUT_0_8.npz"))
hm = heur.predict(t) > 0
lm = lgbm.predict(t) > 0
gt = t["forest_gt_pre2020"] > 0
missed_by_heur = gt & ~hm
print(f"\n48PUT_0_8 — heur missed {missed_by_heur.sum():,} alerts; "
      f"lgbm catches {(missed_by_heur & lm).sum():,} of them "
      f"({(missed_by_heur & lm).sum() / max(missed_by_heur.sum(),1):.1%})")

# Tier distribution — useful for deciding downstream loss weighting
tiers = lgbm.predict_tiers(t)
for name, code in [("NON", 0), ("UNC", 1), ("SOFT", 2), ("STRONG", 3)]:
    print(f"  tier {name}: {(tiers == code).mean():.3f}")


/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/sklea

tile            heur_r  heur_c   lgbm_r  lgbm_c
18NWG_6_6        0.962   0.842    1.000   0.995
18NWH_1_4        0.906   0.884    1.000   0.999
18NXH_6_8        0.999   0.986    1.000   0.999
18NXJ_7_6        0.998   0.793    1.000   0.996
48PUT_0_8        0.824   0.534    0.996   0.918
48PWV_7_8        0.993   0.805    1.000   0.936
48PXC_7_7        0.974   0.654    1.000   0.886
48PYB_3_6        0.974   0.778    1.000   0.930
48QVE_3_0        0.968   0.600    1.000   0.902
48QWD_2_2        0.967   0.564    1.000   0.901

Heuristic agg: recall=0.973  coverage=0.743
Learned   agg: recall=1.000  coverage=0.946


/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



48PUT_0_8 — heur missed 13,064 alerts; lgbm catches 12,739 of them (97.5%)


/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  tier NON: 0.079
  tier UNC: 0.005
  tier SOFT: 0.006
  tier STRONG: 0.910


### Model has no middle ground; only learned decision boundary (what became alert pixel) pasture, cropland, shrubland, savanna, young regrowth were dropped -> when classifying non forrest pixels, they are still closer to forrest than bare -> false positive 

## Test after changed negative-sampling logic

In [1]:
import numpy as np
from pathlib import Path
from model import HeuristicForestMasker, LearnedForestMasker, evaluate_masker_on_tiles

paths = sorted(Path("../cache").glob("*.npz"))
tensors_list = [dict(np.load(p)) for p in paths]

heur = HeuristicForestMasker(ndvi_threshold=0.55, nbr_threshold=0.3, max_ndvi_std=None)
lgbm = LearnedForestMasker(backend="lightgbm").fit(tensors_list)

h = evaluate_masker_on_tiles(heur, paths)
l = evaluate_masker_on_tiles(lgbm, paths)

print(f"{'tile':<14} {'heur_r':>7} {'heur_c':>7}  {'lgbm_r':>7} {'lgbm_c':>7}")
for tid in h["per_tile"]:
    hp, lp = h["per_tile"][tid], l["per_tile"][tid]
    print(f"{tid:<14} {hp['recall_vs_alerts']:>7.3f} {hp['mask_coverage']:>7.3f}  "
          f"{lp['recall_vs_alerts']:>7.3f} {lp['mask_coverage']:>7.3f}")

print(f"\nHeuristic agg: recall={h['aggregate']['recall_vs_alerts']:.3f}  "
      f"coverage={h['aggregate']['mask_coverage']:.3f}")
print(f"Learned   agg: recall={l['aggregate']['recall_vs_alerts']:.3f}  "
      f"coverage={l['aggregate']['mask_coverage']:.3f}")

# Sanity on the weak tile: does lgbm rescue the pixels heuristic missed?
t = dict(np.load("../cache/48PUT_0_8.npz"))
hm = heur.predict(t) > 0
lm = lgbm.predict(t) > 0
gt = t["forest_gt_pre2020"] > 0
missed_by_heur = gt & ~hm
print(f"\n48PUT_0_8 — heur missed {missed_by_heur.sum():,} alerts; "
      f"lgbm catches {(missed_by_heur & lm).sum():,} of them "
      f"({(missed_by_heur & lm).sum() / max(missed_by_heur.sum(),1):.1%})")

# Tier distribution — useful for deciding downstream loss weighting
tiers = lgbm.predict_tiers(t)
for name, code in [("NON", 0), ("UNC", 1), ("SOFT", 2), ("STRONG", 3)]:
    print(f"  tier {name}: {(tiers == code).mean():.3f}")


/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/sklea

tile            heur_r  heur_c   lgbm_r  lgbm_c
18NWG_6_6        0.962   0.842    0.967   0.852
18NWH_1_4        0.906   0.884    0.943   0.899
18NXH_6_8        0.999   0.986    0.999   0.986
18NXJ_7_6        0.998   0.793    0.999   0.824
48PUT_0_8        0.824   0.534    0.934   0.572
48PWV_7_8        0.993   0.805    0.997   0.843
48PXC_7_7        0.974   0.654    0.984   0.742
48PYB_3_6        0.974   0.778    0.986   0.810
48QVE_3_0        0.968   0.600    0.971   0.621
48QWD_2_2        0.967   0.564    0.972   0.604

Heuristic agg: recall=0.973  coverage=0.743
Learned   agg: recall=0.982  coverage=0.775


/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



48PUT_0_8 — heur missed 13,064 alerts; lgbm catches 8,195 of them (62.7%)


/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  tier NON: 0.402
  tier UNC: 0.036
  tier SOFT: 0.015
  tier STRONG: 0.547
